In [ ]:
env_name = dbutils.widgets.get("env") if dbutils.widgets.get("env") else "dev"
# pipeline_id = dbutils.widgets.get("pipeline_id") if dbutils.widgets.get("pipeline_id") else "placeholder"
# job_run_id = dbutils.widgets.get("job_run_id") if dbutils.widgets.get("job_run_id") else "placeholder"
# task_id = dbutils.widgets.get("task_run_id") if dbutils.widgets.get("task_run_id") else "placeholder"

catalog_name = f"citibike_ext_{env_name}"
checkpoint_path = f"/Volumes/{catalog_name}/03_gold/_checkpoint/gold_citibikes/"
table_name = f"{catalog_name}.02_silver.citibike_trips"

In [ ]:
silver_table = f"{catalog_name}.02_silver.citibike_trips"
gold_table = f"{catalog_name}.03_gold.daily_ride_summary"
checkpoint = f"/Volumes/{catalog_name}/03_gold/_checkpoint/daily_ride_summary"


def recompute_affected_days(batch_df, batch_id):
    spark = batch_df.sparkSession

    # 1. Distinct days touched by this batch of changes.
    #    Note: we do NOT filter _change_type and do NOT drop pre-images here.
    affected = batch_df.select("trip_start_date").distinct()
    if affected.isEmpty():  # DataFrame.isEmpty() is Spark 3.3+
        return
    affected.createOrReplaceTempView("affected_days")

    # 2. Recompute those days IN FULL from current silver — this is what makes
    #    max/min correct and handles deletes, because we aggregate live rows.
    spark.sql(f"""
        CREATE OR REPLACE TEMP VIEW recomputed AS
        SELECT s.trip_start_date,
               round(max(s.trip_duration_mins), 2) AS max_trip_duration_mins,
               round(min(s.trip_duration_mins), 2) AS min_trip_duration_mins,
               round(avg(s.trip_duration_mins), 2) AS avg_trip_duration_mins,
               count(*)                     AS total_trips
        FROM {silver_table} s
        WHERE s.trip_start_date IN (SELECT trip_start_date FROM affected_days)
        GROUP BY s.trip_start_date
    """)

    # 3. Merge recomputed day-summaries into gold.
    spark.sql(f"""
        MERGE INTO {gold_table} t
        USING recomputed s
        ON t.trip_start_date = s.trip_start_date
        WHEN MATCHED THEN UPDATE SET *
        WHEN NOT MATCHED THEN INSERT *
    """)


spark.sql(f"""
CREATE TABLE IF NOT EXISTS {gold_table} (
    trip_start_date        DATE,
    max_trip_duration_mins DOUBLE,
    min_trip_duration_mins DOUBLE,
    avg_trip_duration_mins DOUBLE,
    total_trips            BIGINT
) TBLPROPERTIES (delta.enableChangeDataFeed = true);
""")

(
    spark.readStream.format("delta")
    .option("readChangeFeed", "true")
    .table(silver_table)
    .writeStream.option("checkpointLocation", checkpoint)
    .trigger(availableNow=True)
    .foreachBatch(recompute_affected_days)
    .start()
)